In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import re
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ============================================================
# User configuration
# ============================================================

root_dir = Path(os.getcwd())

dataset_names = [
    "Replogle_K562_essential",
    "Replogle_RPE",
    "NormanWeissman2019",
    "ChangYe",
    "ZhaoSims2021",
]

groups = ["single", "dual", "multi"]

In [ ]:
# If True, process every existing dataset/group under root_dir.
# If False, only process SELECTED_SUB_PATH.
RUN_ALL_VALID_PATHS = True
SELECTED_SUB_PATH: Path | None = None

# Selected variants are read from:
#   <dataset>_pseudo_pairing_evaluation/<group>/result_analysis/selected_variants_TEMPLATE_EDIT_ME.csv
SELECTION_TABLE_NAME = "selected_variants_TEMPLATE_EDIT_ME.csv"

# S0 is shown as a dashed reference line, not a bar, by default.
INCLUDE_S0_AS_BAR = False
NAIVE_BASELINE_ID = "S0_naive_mean_control_reference"
SHOW_NAIVE_BASELINE_LINE = True
NAIVE_BASELINE_LINE_COLOR = "#EE5862"
NAIVE_BASELINE_LINESTYLE = "--"
NAIVE_BASELINE_LINEWIDTH = 1.15
NAIVE_BASELINE_LINE_ALPHA = 0.85
NAIVE_BASELINE_TEXT = "Naive average control"
NAIVE_BASELINE_TEXT_SIZE = 9
NAIVE_BASELINE_TEXT_COLOR = "#555555"
NAIVE_BASELINE_TEXT_X_FRACTION = 0.985
NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION = 0.018

# Bar appearance.
FIGSIZE = (10, 6)
BAR_WIDTH = 0.85
BAR_ALPHA = 0.80
ERRORBAR_COLOR = "#202020"
ERRORBAR_LINEWIDTH = 1.0
ERRORBAR_CAPSIZE = 5

ROTATE_XTICKS = 0
GRID_ALPHA = 0.28
DPI = 300
SAVE_PNG = True
SAVE_SVG = True
SHOW_FIGURES = True

# Exact mean-value labels above each bar.
SHOW_BAR_VALUE_LABELS = True
BAR_VALUE_FONT_SIZE = 8.0
BAR_VALUE_OFFSET_FRACTION = 0.025
BAR_VALUE_COLOR = "#222222"

# Per-seed scatter overlay.
SHOW_SEED_SCATTERS = True
POINT_JITTER = 0.165
POINT_SIZE = 31
POINT_ALPHA = 0.88
POINT_EDGE_COLOR = "black"
POINT_EDGE_WIDTH = 0.45
POINT_RANDOM_SEED = 123
DRAW_SEED_LINES = False

# Significance testing: each selected strategy is compared against random single control.
REFERENCE_STRATEGY_FOR_TEST = "S1_random_single_control"
REFERENCE_STRATEGY_FOR_TEST_LABEL = "Random single control"
REFERENCE_BAR_TEXT = "ref"
SHOW_REFERENCE_BAR_TEXT = False
STAT_MODE = "paired"  # "auto", "paired", or "unpaired"
ALPHA = 0.05
PAIRWISE_CORRECTION = "holm"
SHOW_PAIRWISE_STAR_ANNOTATIONS = True
STAR_ANNOTATE_NS = True
STAR_FONT_SIZE = 10.0
STAR_FONT_WEIGHT = "bold"
STAR_COLOR = "#222222"
STAR_TEXT_OFFSET_FRACTION = 0.045
STAR_Y_EXTRA_FRACTION = 0.18
SHOW_SIGNIFICANCE_NOTE = True

# Seed-level perturbation-effect table from the original all-gene evaluation pipeline.
PERTURBATION_EFFECT_SEED_TABLE = Path("perturbation_effect") / "perturbation_effect_consistency_repeated_run_summary.csv"
PERTURBATION_EFFECT_SEED_TABLE_PARTIAL = Path("perturbation_effect") / "perturbation_effect_consistency_repeated_run_summary_PARTIAL.csv"

# Stabilize y-axis when metric values are very close.
MIN_RELATIVE_SPAN = 0.08
MIN_ABSOLUTE_SPAN = 1e-4
PEARSON_MIN_SPAN = 0.05
TOP_PADDING_FRACTION = 0.16
BOTTOM_PADDING_FRACTION = 0.08

# Combined multi-dataset figures.
MAKE_COMBINED_FIGURES = True
COMBINED_NCOLS = 3
COMBINED_FIGSIZE_PER_PANEL = (8, 5)

# Output folders.
INDIVIDUAL_OUTPUT_FOLDER_NAME = "perturbation_effect_selected_strategy_metric_barplots"
COMBINED_OUTPUT_FOLDER_NAME = "perturbation_effect_selected_strategy_metric_combined"

# Metrics requested: all-gene-level perturbation effect metrics.
# Values are read directly from selected_variants_TEMPLATE_EDIT_ME.csv.
PERTURBATION_EFFECT_METRICS = {
    "perturbation_effect_rmse": {
        "label": "All-gene perturbation-effect RMSE",
        "ylabel": "RMSE",
        "direction": "lower",
        "save_name": "perturbation_effect_rmse",
        "mean_column": "perturbation_effect__perturbation_effect_rmse_mean",
        "std_column": "perturbation_effect__perturbation_effect_rmse_std",
        "n_column": "perturbation_effect__perturbation_effect_rmse_n",
        "seed_source_columns": [
            "strategy_delta_rmse_common_mean",
            "perturbation_effect_rmse",
            "perturbation_effect__perturbation_effect_rmse_mean",
        ],
    },
    # "perturbation_effect_pearson": {
    #     "label": "All-gene perturbation-effect Pearson",
    #     "ylabel": "Pearson correlation",
    #     "direction": "higher",
    #     "save_name": "perturbation_effect_pearson",
    #     "mean_column": "perturbation_effect__perturbation_effect_pearson_mean",
    #     "std_column": "perturbation_effect__perturbation_effect_pearson_std",
    #     "n_column": "perturbation_effect__perturbation_effect_pearson_n",
    #     "seed_source_columns": [
    #         "strategy_delta_pearson_common_mean",
    #         "perturbation_effect_pearson",
    #         "perturbation_effect__perturbation_effect_pearson_mean",
    #     ],
    # },
}


# ============================================================
# Strategy labels and colors
# ============================================================

STRATEGY_PLOT_LABELS = {
    "S0_naive_mean_control_reference": "Naive\nmean\ncontrol",
    "S1_random_single_control": "Random\nsingle\ncontrol",
    "S2_random_average_controls": "Random\naverage\ncontrol",
    "S4_SEACell_balanced_random_sample": "Metacell\nbalanced\nrandom",
    "S3_SEACell_metacell_average": "Random\nmetacell\naverage",
    "S5_SEACell_OT_sampled_average": "Metacell OT\nsampled\naverage",
}

STRATEGY_BASE_COLORS = {
    "S0_naive_mean_control_reference": "#D0E0EF",
    "S1_random_single_control": "#6E8FB2",
    "S2_random_average_controls": "#7DA494",
    "S3_SEACell_metacell_average": "#E5A79A",
    "S4_SEACell_balanced_random_sample": "#EAB67A",
    "S5_SEACell_OT_sampled_average": "#9F8DB8",
}

S5_VARIANT_COLORS = {
    "200&5": "#D49AB5",
    "350&5": "#9F8DB8",
    "500&5": "#B66699",
}

STRATEGY_VARIANT_COLOR_POOLS = {
    "S3_SEACell_metacell_average": ["#8EBCBB", "#74AAA9", "#5E9796", "#A8CECD"],
    "S4_SEACell_balanced_random_sample": ["#68A6A4", "#4F8F8D", "#3B7775", "#8CBDBB"],
    "S5_SEACell_OT_sampled_average": ["#D49AB5", "#B66699", "#8F3E77", "#E8B8CB"],
}

DEFAULT_STRATEGY_RENAME_MAP = {
    "S0_naive_mean_control_reference": "S0_naive_mean_control_reference",
    "S1_random_single_control": "S1_random_single_control",
    "S2_random_average_controls": "S2_random_average_controls",
    "S3_SEACell_metacell_average": "S3_SEACell_metacell_average",
    "S4_SEACell_balanced_random_sample": "S4_SEACell_balanced_random_sample",
    "S5_SEACell_OT_sampled_average": "S5_SEACell_OT_sampled_average",
    "S0": "S0_naive_mean_control_reference",
    "S1": "S1_random_single_control",
    "S2": "S2_random_average_controls",
    "S3": "S3_SEACell_metacell_average",
    "S4": "S4_SEACell_balanced_random_sample",
    "S5": "S5_SEACell_OT_sampled_average",
    "S4_random_single_control_oracle": "S1_random_single_control",
    "S4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control": "S1_random_single_control",
    "strategy4_random_single_control_cell": "S1_random_single_control",
    "S3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_controls": "S2_random_average_controls",
    "strategy3_random_average_control_cells": "S2_random_average_controls",
    "S5_random_metacell_average": "S3_SEACell_metacell_average",
    "S3_random_metacell_average": "S3_SEACell_metacell_average",
    "strategy5_random_metacell_average": "S3_SEACell_metacell_average",
    "S1_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "S4_SEACell_balanced_random": "S4_SEACell_balanced_random_sample",
    "strategy1_seacell_balanced_random_repeated": "S4_SEACell_balanced_random_sample",
    "S2_SEACell_OT_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "S2_SEACell_OT_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
    "strategy2_seacells_ot_topk_sampled_average": "S5_SEACell_OT_sampled_average",
    "strategy2_seacell_ot_topk_sampled_average_repeated": "S5_SEACell_OT_sampled_average",
}

STRATEGY_ORDER_MAP = {
    "S0_naive_mean_control_reference": 0,
    "S1_random_single_control": 1,
    "S2_random_average_controls": 2,
    "S3_SEACell_metacell_average": 3,
    "S4_SEACell_balanced_random_sample": 4,
    "S5_SEACell_OT_sampled_average": 5,
}

In [ ]:
# ============================================================
# Basic IO and formatting helpers
# ============================================================

def read_table(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t")
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table format: {path}")


def as_bool_series(s: pd.Series) -> pd.Series:
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y"])


def is_missing(x: Any) -> bool:
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def is_valid_color(x: Any) -> bool:
    if is_missing(x):
        return False
    x = str(x).strip()
    return bool(x) and x.lower() not in {"nan", "none", "null"}


def is_finite_number(value: Any) -> bool:
    try:
        return pd.notna(value) and str(value).strip() != "" and np.isfinite(float(value))
    except Exception:
        return False


def fmt_int_like(x: Any) -> str:
    if is_missing(x):
        return "NA"
    try:
        x = float(x)
        return str(int(x)) if x.is_integer() else f"{x:g}"
    except Exception:
        return str(x)


def clean_label(label: str) -> str:
    return " ".join(str(label).replace("\n", " ").split())


def final_bar_value(value: Any, digits: int = 4) -> str:
    if is_missing(value):
        return ""
    value = float(value)
    if abs(value) >= 100:
        return f"{value:.0f}"
    if abs(value) >= 10:
        return f"{value:.1f}"
    if abs(value) >= 1:
        return f"{value:.2f}"
    if abs(value) >= 0.01:
        return f"{value:.{digits}f}"
    return f"{value:.2e}"


def extract_variant_suffix(display_label: str) -> str:
    display_label = str(display_label)
    if "(" in display_label and ")" in display_label:
        return display_label[display_label.rfind("("):].strip()
    return ""


def extract_number_from_text(x: Any, patterns: Iterable[str]) -> float:
    text = str(x)
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            try:
                return float(m.group(1))
            except Exception:
                pass
    return np.nan


def safe_filename(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x)).strip("_")


def dataset_group_from_path(path: Path) -> tuple[str, str]:
    dataset = path.parent.name.replace("_pseudo_pairing_evaluation", "")
    group = path.name
    return dataset, group


def get_dataset_group_title(path: Path) -> str:
    dataset, group = dataset_group_from_path(path)
    return f"{dataset} | {group}"


# ============================================================
# Variant metadata harmonization
# ============================================================

def fill_numeric_from_candidates(df: pd.DataFrame, target: str, candidates: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in candidates:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors="coerce")
            df[target] = df[target].where(df[target].notna(), vals)


def fill_from_text_patterns(df: pd.DataFrame, target: str, text_cols: list[str], patterns: list[str]) -> None:
    if target not in df.columns:
        df[target] = np.nan
    df[target] = pd.to_numeric(df[target], errors="coerce")
    for col in text_cols:
        if col not in df.columns:
            continue
        vals = df[col].map(lambda x: extract_number_from_text(x, patterns))
        df[target] = df[target].where(df[target].notna(), vals)


def infer_strategy_column(df: pd.DataFrame) -> str:
    for col in ["strategy", "strategy_id", "pairing_strategy"]:
        if col in df.columns:
            return col
    raise KeyError(f"Cannot infer strategy column from columns: {list(df.columns)}")


def make_variant_label(row: pd.Series | dict[str, Any]) -> str:
    strategy = str(row.get("strategy", ""))
    nmc = row.get("n_metacells", np.nan)
    topk = row.get("top_k", np.nan)
    sampled = row.get("sampled_metacells_k", np.nan)
    k_avg = row.get("n_control_cells_to_average", np.nan)

    if strategy in {"S0_naive_mean_control_reference", "S1_random_single_control"}:
        return "default"

    if strategy == "S2_random_average_controls":
        return "default" if is_missing(k_avg) else f"k_{fmt_int_like(k_avg)}"

    if strategy == "S3_SEACell_metacell_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(sampled):
            parts.append(f"sampledMC_{fmt_int_like(sampled)}")
        return "__".join(parts) if parts else "default"

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"nmc_{fmt_int_like(nmc)}" if not is_missing(nmc) else "default"

    if strategy == "S5_SEACell_OT_sampled_average":
        parts = []
        if not is_missing(nmc):
            parts.append(f"nmc_{fmt_int_like(nmc)}")
        if not is_missing(topk):
            parts.append(f"topk_{fmt_int_like(topk)}")
        return "__".join(parts) if parts else "default"

    return "default"


def make_variant_match_key(row: pd.Series | dict[str, Any]) -> str:
    """Build a table-agnostic variant key from strategy-defining parameters.

    This avoids mismatches where the selected table and seed-level result table
    use different variant_id strings for the same strategy setting, e.g.
    sampledMC_10 versus k_10, or S2 without a suffix versus S2__k_100.
    """
    strategy = str(row.get("strategy", ""))
    label = make_variant_label(row)
    if label == "default":
        return strategy
    return f"{strategy}__{label}"


def canonicalize_variant_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    sc_col = infer_strategy_column(out)
    out["strategy_old"] = out[sc_col].astype(str)
    out["strategy"] = out["strategy_old"].map(DEFAULT_STRATEGY_RENAME_MAP).fillna(out["strategy_old"])

    if "strategy_order" not in out.columns:
        out["strategy_order"] = out["strategy"].map(STRATEGY_ORDER_MAP).fillna(99).astype(int)
    else:
        out["strategy_order"] = pd.to_numeric(out["strategy_order"], errors="coerce")
        out["strategy_order"] = out["strategy_order"].where(out["strategy_order"].notna(), out["strategy"].map(STRATEGY_ORDER_MAP))
        out["strategy_order"] = out["strategy_order"].fillna(99).astype(int)

    fill_numeric_from_candidates(out, "n_metacells", ["n_metacells", "n_metacells_requested", "n_metacells_observed"])
    fill_numeric_from_candidates(out, "top_k", ["top_k", "top_k_metacells"])
    fill_numeric_from_candidates(out, "sampled_metacells_k", ["sampled_metacells_k", "n_metacells_to_average"])
    fill_numeric_from_candidates(out, "n_control_cells_to_average", ["n_control_cells_to_average"])

    text_cols = [
        c
        for c in ["variant_id", "variant_label", "display_variant_label", "parameter_label", "outdir", "pseudo_control_h5ad"]
        if c in out.columns
    ]
    fill_from_text_patterns(out, "n_metacells", text_cols, [r"nmc[_= -]?(\d+)", r"metacells[_= -]?(\d+)"])
    fill_from_text_patterns(out, "top_k", text_cols, [r"topk[_= -]?(\d+)", r"top_k[_= -]?(\d+)"])
    fill_from_text_patterns(out, "sampled_metacells_k", text_cols, [r"sampledMC[_= -]?(\d+)", r"sampled[_= -]?metacells[_= -]?(\d+)"])

    if "variant_id" not in out.columns:
        out["variant_label"] = [make_variant_label(row) for _, row in out.iterrows()]
        out["variant_id"] = out["strategy"].astype(str) + "__" + out["variant_label"].astype(str)
    else:
        out["variant_id"] = out["variant_id"].astype(str)

    # Preserve the source table id and create a robust key used for matching
    # selected variants to seed-level result rows.
    out["source_variant_id"] = out["variant_id"].astype(str)
    out["variant_match_key"] = [make_variant_match_key(row) for _, row in out.iterrows()]

    if "display_variant_label" not in out.columns:
        out["display_variant_label"] = out["variant_id"].astype(str)

    return out


def assign_plot_colors(selected: pd.DataFrame) -> pd.Series:
    colors: list[str] = []
    strategy_counts: dict[str, int] = {}

    for _, row in selected.iterrows():
        for col in ["manual_color", "color"]:
            if col in row.index and is_valid_color(row[col]):
                colors.append(str(row[col]).strip())
                break
        else:
            strategy = str(row.get("strategy", ""))
            display = str(row.get("display_variant_label", ""))

            if strategy == "S5_SEACell_OT_sampled_average":
                chosen = None
                for key, color in S5_VARIANT_COLORS.items():
                    if key in display:
                        chosen = color
                        break
                if chosen is not None:
                    colors.append(chosen)
                    continue

            idx = strategy_counts.get(strategy, 0)
            strategy_counts[strategy] = idx + 1
            pool = STRATEGY_VARIANT_COLOR_POOLS.get(strategy)

            if pool and selected["strategy"].astype(str).eq(strategy).sum() > 1:
                colors.append(pool[idx % len(pool)])
            else:
                colors.append(STRATEGY_BASE_COLORS.get(strategy, "#999999"))

    return pd.Series(colors, index=selected.index)


def variant_suffix_for_label(row: pd.Series) -> str:
    strategy = str(row.get("strategy", ""))

    nmc = row.get("n_metacells", np.nan)
    if not is_finite_number(nmc):
        nmc = row.get("n_metacells_observed", np.nan)

    if strategy == "S2_random_average_controls":
        k = row.get("n_control_cells_to_average", np.nan)
        return f"\n(k={fmt_int_like(k)})" if is_finite_number(k) else ""

    if strategy == "S3_SEACell_metacell_average":
        k = row.get("sampled_metacells_k", row.get("n_metacells_to_average", np.nan))
        if is_finite_number(nmc) and is_finite_number(k):
            return f"\n({fmt_int_like(nmc)}&{fmt_int_like(k)})"
        if is_finite_number(nmc):
            return f"\n({fmt_int_like(nmc)})"
        return ""

    if strategy == "S4_SEACell_balanced_random_sample":
        return f"\n({fmt_int_like(nmc)})" if is_finite_number(nmc) else ""

    if strategy == "S5_SEACell_OT_sampled_average":
        topk = row.get("top_k", row.get("top_k_metacells", np.nan))
        if is_finite_number(nmc) and is_finite_number(topk):
            return f"\n({fmt_int_like(nmc)}&{fmt_int_like(topk)})"
        if is_finite_number(nmc):
            return f"\n({fmt_int_like(nmc)})"
        if is_finite_number(topk):
            return f"\n(topk={fmt_int_like(topk)})"
        return ""

    suffix = extract_variant_suffix(str(row.get("display_variant_label", "")))
    return f"\n{suffix}" if suffix else ""


def make_bar_xtick_label(row: pd.Series, n: int | float | None = None, show_n: bool = True) -> str:
    strategy = str(row.get("strategy", ""))
    base = STRATEGY_PLOT_LABELS.get(strategy, clean_label(str(row.get("display_variant_label", strategy))))
    label = f"{base}{variant_suffix_for_label(row)}"
    if show_n and n is not None and is_finite_number(n):
        label = f"{label}\nn={int(float(n))}"
    return label


# ============================================================
# Selection-table loading
# ============================================================

def candidate_sub_paths() -> list[Path]:
    if not RUN_ALL_VALID_PATHS:
        if SELECTED_SUB_PATH is None:
            raise ValueError("SELECTED_SUB_PATH must be set when RUN_ALL_VALID_PATHS=False.")
        return [Path(SELECTED_SUB_PATH)]

    paths = []
    for dataset_name in dataset_names:
        base = root_dir / f"{dataset_name}_pseudo_pairing_evaluation"
        for group in groups:
            sub = base / group
            if (sub / "result_analysis" / SELECTION_TABLE_NAME).exists():
                paths.append(sub)
    return paths


def filter_selection_to_context(df: pd.DataFrame, sub_path: Path) -> pd.DataFrame:
    dataset, group = dataset_group_from_path(sub_path)
    out = df.copy()

    # Most selected-variant tables are already per dataset/group. This filter is only
    # applied when the table contains multiple dataset/group entries.
    if "dataset_id" in out.columns:
        matching = out["dataset_id"].astype(str).eq(dataset)
        if matching.any():
            out = out[matching].copy()

    if "perturbed_group" in out.columns:
        matching = out["perturbed_group"].astype(str).eq(group)
        if matching.any():
            out = out[matching].copy()

    return out


def load_selection_table(selection_path: Path, sub_path: Path) -> pd.DataFrame:
    selection = read_table(selection_path)
    selection = filter_selection_to_context(selection, sub_path)
    selection = canonicalize_variant_table(selection)

    if "select_for_final" not in selection.columns:
        raise KeyError(f"Selection table lacks 'select_for_final': {selection_path}")

    # Coerce metric columns used here.
    for info in PERTURBATION_EFFECT_METRICS.values():
        for col in [info["mean_column"], info["std_column"], info["n_column"]]:
            if col in selection.columns:
                selection[col] = pd.to_numeric(selection[col], errors="coerce")

    selected = selection[as_bool_series(selection["select_for_final"])].copy()
    if not INCLUDE_S0_AS_BAR:
        selected = selected[selected["strategy"].astype(str) != NAIVE_BASELINE_ID].copy()

    if selected.empty:
        raise RuntimeError(f"No selected variants found in {selection_path}")

    selected["_selection_order"] = np.arange(selected.shape[0])
    selected = selected.sort_values(["_selection_order"]).reset_index(drop=True)
    selected["plot_color"] = assign_plot_colors(selected)

    return selected, selection


def selected_metric_context(sub_path: Path) -> dict[str, Any]:
    selection_path = sub_path / "result_analysis" / SELECTION_TABLE_NAME
    if not selection_path.exists():
        raise FileNotFoundError(f"Missing selection table: {selection_path}")

    selected, full_selection = load_selection_table(selection_path, sub_path)
    seed_df, seed_path = load_seed_level_perturbation_effect_table(sub_path, selected, full_selection)

    return {
        "sub_path": sub_path,
        "dataset_group_title": get_dataset_group_title(sub_path),
        "selection_path": selection_path,
        "selected": selected,
        "full_selection": full_selection,
        "seed_df": seed_df,
        "seed_path": seed_path,
    }


In [ ]:
# ============================================================
# Seed-level perturbation-effect loading and statistical testing
# ============================================================

def first_existing_path(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def add_sampling_seed_for_plot(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    seed_col = None
    for col in ["sampling_seed", "seed", "pair_selection_seed", "random_seed", "run_seed"]:
        if col in out.columns and pd.to_numeric(out[col], errors="coerce").notna().any():
            seed_col = col
            break

    if seed_col is not None:
        out["sampling_seed_for_plot"] = pd.to_numeric(out[seed_col], errors="coerce")
    else:
        # Fallback: parse seed-like suffixes from run_id / parameter_label / paths.
        out["sampling_seed_for_plot"] = np.nan
        text_cols = [
            c
            for c in ["run_id", "parameter_label", "variant_id", "source_variant_id", "pseudo_control_h5ad", "outdir"]
            if c in out.columns
        ]
        for col in text_cols:
            parsed = out[col].astype(str).str.extract(r"(?:seed|sampling_seed|pair_seed)[_= -]?(\d+)", expand=False)
            parsed = pd.to_numeric(parsed, errors="coerce")
            out["sampling_seed_for_plot"] = out["sampling_seed_for_plot"].where(out["sampling_seed_for_plot"].notna(), parsed)

    # S0 usually has only one deterministic row. Give it a stable seed id if absent.
    s0_mask = out["strategy"].astype(str).eq(NAIVE_BASELINE_ID)
    out.loc[s0_mask & out["sampling_seed_for_plot"].isna(), "sampling_seed_for_plot"] = -1

    # Remaining missing seeds are assigned within each variant so scatter points still display.
    missing = out["sampling_seed_for_plot"].isna()
    if missing.any():
        out.loc[missing, "sampling_seed_for_plot"] = (
            out[missing]
            .groupby("variant_match_key", dropna=False)
            .cumcount()
            .astype(float)
        )

    return out


def fill_seed_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for metric, info in PERTURBATION_EFFECT_METRICS.items():
        if metric not in out.columns:
            out[metric] = np.nan
        out[metric] = pd.to_numeric(out[metric], errors="coerce")
        for col in info.get("seed_source_columns", []):
            if col in out.columns:
                vals = pd.to_numeric(out[col], errors="coerce")
                out[metric] = out[metric].where(out[metric].notna(), vals)
    return out


def load_seed_level_perturbation_effect_table(
    sub_path: Path,
    selected: pd.DataFrame,
    full_selection: pd.DataFrame,
) -> tuple[pd.DataFrame, Path | None]:
    seed_path = first_existing_path([
        sub_path / PERTURBATION_EFFECT_SEED_TABLE,
        sub_path / PERTURBATION_EFFECT_SEED_TABLE_PARTIAL,
    ])

    if seed_path is None:
        print(f"[Warning] Missing seed-level perturbation-effect table under {sub_path}; using selected-table means only.")
        return pd.DataFrame(), None

    seed_df = read_table(seed_path)
    seed_df = filter_selection_to_context(seed_df, sub_path)
    seed_df = canonicalize_variant_table(seed_df)
    seed_df = add_sampling_seed_for_plot(seed_df)
    seed_df = fill_seed_metric_columns(seed_df)

    # Keep selected variants, S0 baseline, and S1 reference for tests even if S1 is not selected as a bar.
    wanted_keys = set(selected["variant_match_key"].astype(str))
    wanted_keys.add(NAIVE_BASELINE_ID)
    wanted_keys.add(REFERENCE_STRATEGY_FOR_TEST)
    seed_df = seed_df[seed_df["variant_match_key"].astype(str).isin(wanted_keys)].copy()

    if seed_df.empty:
        print(f"[Warning] Seed table exists but no rows match selected variants: {seed_path}")
        return pd.DataFrame(), seed_path

    # If the result table has multiple rows per variant/seed, reduce to one seed-level value.
    group_cols = ["variant_match_key", "sampling_seed_for_plot"]
    agg: dict[str, str] = {metric: "mean" for metric in PERTURBATION_EFFECT_METRICS if metric in seed_df.columns}
    for col in [
        "strategy", "strategy_order", "source_variant_id", "variant_id", "display_variant_label",
        "n_metacells", "top_k", "sampled_metacells_k", "n_control_cells_to_average",
    ]:
        if col in seed_df.columns:
            agg[col] = "first"

    seed_df = seed_df.groupby(group_cols, dropna=False, as_index=False).agg(agg)
    return seed_df, seed_path


def seed_values_for_order(
    selected: pd.DataFrame,
    seed_df: pd.DataFrame,
    metric: str,
) -> tuple[list[np.ndarray], np.ndarray, np.ndarray, np.ndarray, bool]:
    """Return values/mean/std/n in selected order.

    If seed-level rows are available for a variant, bars are computed from seed
    values. Otherwise the aggregated mean/std/n from selected_variants is used.
    """
    means_from_table, stds_from_table, ns_from_table = extract_metric_arrays(selected, metric)
    y_values: list[np.ndarray] = []
    means: list[float] = []
    stds: list[float] = []
    ns: list[float] = []
    used_seed_level = False

    for i, row in selected.reset_index(drop=True).iterrows():
        key = str(row["variant_match_key"])
        vals = np.array([], dtype=float)
        if not seed_df.empty and metric in seed_df.columns:
            vals = pd.to_numeric(
                seed_df.loc[seed_df["variant_match_key"].astype(str).eq(key), metric],
                errors="coerce",
            ).dropna().to_numpy(dtype=float)

        if len(vals) > 0:
            used_seed_level = True
            y_values.append(vals)
            means.append(float(np.nanmean(vals)))
            stds.append(float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan)
            ns.append(float(len(vals)))
        else:
            mean = means_from_table[i]
            std = stds_from_table[i]
            n = ns_from_table[i]
            y_values.append(np.array([mean], dtype=float) if np.isfinite(mean) else np.array([], dtype=float))
            means.append(float(mean) if np.isfinite(mean) else np.nan)
            stds.append(float(std) if np.isfinite(std) else np.nan)
            ns.append(float(n) if np.isfinite(n) else np.nan)

    return y_values, np.asarray(means, dtype=float), np.asarray(stds, dtype=float), np.asarray(ns, dtype=float), used_seed_level


def extract_seed_baseline(seed_df: pd.DataFrame, metric: str) -> tuple[float, float, float, np.ndarray]:
    if seed_df.empty or metric not in seed_df.columns:
        return np.nan, np.nan, np.nan, np.array([], dtype=float)
    vals = pd.to_numeric(
        seed_df.loc[seed_df["variant_match_key"].astype(str).eq(NAIVE_BASELINE_ID), metric],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)
    if len(vals) == 0:
        return np.nan, np.nan, np.nan, vals
    return (
        float(np.nanmean(vals)),
        float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else np.nan,
        float(len(vals)),
        vals,
    )


def infer_pairwise_mode(data: pd.DataFrame, metric: str, reference_key: str, candidate_key: str, stat_mode: str) -> tuple[bool, pd.DataFrame]:
    ref = data.loc[data["variant_match_key"].astype(str).eq(reference_key), ["sampling_seed_for_plot", metric]].dropna()
    cand = data.loc[data["variant_match_key"].astype(str).eq(candidate_key), ["sampling_seed_for_plot", metric]].dropna()
    ref_by_seed = ref.pivot_table(index="sampling_seed_for_plot", values=metric, aggfunc="mean")
    cand_by_seed = cand.pivot_table(index="sampling_seed_for_plot", values=metric, aggfunc="mean")
    paired_frame = ref_by_seed.join(cand_by_seed, how="inner", lsuffix="_ref", rsuffix="_cand").dropna()

    if stat_mode == "paired":
        return True, paired_frame
    if stat_mode == "unpaired":
        return False, paired_frame
    return paired_frame.shape[0] >= 2, paired_frame


def holm_adjust(p_values: list[float]) -> list[float]:
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return []
    order = np.argsort(p)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj = (m - rank) * p[idx]
        adj = max(adj, running_max)
        adjusted[idx] = min(adj, 1.0)
        running_max = adjusted[idx]
    return adjusted.tolist()


def p_to_stars(p: float) -> str:
    if pd.isna(p):
        return "ns"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def compute_reference_pairwise_statistics(
    seed_df: pd.DataFrame,
    metric: str,
    selected_keys: list[str],
    reference_key: str = REFERENCE_STRATEGY_FOR_TEST,
    stat_mode: str = STAT_MODE,
) -> tuple[pd.DataFrame, str]:
    if seed_df.empty or metric not in seed_df.columns:
        return pd.DataFrame(), "no seed-level data"

    data = seed_df[["variant_match_key", "sampling_seed_for_plot", metric]].dropna().copy()
    if reference_key not in set(data["variant_match_key"].astype(str)):
        return pd.DataFrame(), f"reference {reference_key} absent"

    if stat_mode not in {"auto", "paired", "unpaired"}:
        raise ValueError("STAT_MODE must be 'auto', 'paired', or 'unpaired'.")

    reference_values = data.loc[data["variant_match_key"].astype(str).eq(reference_key), metric].dropna().to_numpy(dtype=float)
    records: list[dict[str, Any]] = []

    for i, key in enumerate(selected_keys):
        if key == reference_key:
            records.append({
                "metric": metric,
                "reference_group": reference_key,
                "group2": key,
                "group2_index": i,
                "test": "reference",
                "paired": np.nan,
                "n_used": int(len(reference_values)),
                "statistic": np.nan,
                "p_value": np.nan,
                "p_adj_holm": np.nan,
                "significance": REFERENCE_BAR_TEXT,
                "significant": False,
            })
            continue

        candidate_values = data.loc[data["variant_match_key"].astype(str).eq(key), metric].dropna().to_numpy(dtype=float)
        if len(candidate_values) == 0:
            records.append({
                "metric": metric, "reference_group": reference_key, "group2": key, "group2_index": i,
                "test": "missing candidate data", "paired": False, "n_used": 0,
                "statistic": np.nan, "p_value": np.nan,
            })
            continue

        paired_used, paired_frame = infer_pairwise_mode(data, metric, reference_key, key, stat_mode)

        if paired_used:
            n = int(paired_frame.shape[0])
            test = "paired t-test vs random single control"
            if n < 2:
                stat = p = np.nan
            else:
                a = paired_frame[f"{metric}_ref"].to_numpy(dtype=float)
                b = paired_frame[f"{metric}_cand"].to_numpy(dtype=float)
                diff = b - a
                mean_diff = float(np.nanmean(diff))
                diff_sd = float(np.nanstd(diff, ddof=1)) if n > 1 else np.nan
                if np.isfinite(diff_sd) and np.isclose(diff_sd, 0.0) and not np.isclose(mean_diff, 0.0):
                    stat = np.inf if mean_diff > 0 else -np.inf
                    p = 0.0
                elif np.allclose(diff, 0):
                    stat, p = 0.0, 1.0
                else:
                    stat, p = stats.ttest_rel(b, a, nan_policy="omit", alternative="two-sided")
        else:
            a = reference_values
            b = candidate_values
            n = int(min(len(a), len(b)))
            test = "Welch t-test vs random single control"
            if len(a) < 2 or len(b) < 2:
                stat = p = np.nan
            else:
                stat, p = stats.ttest_ind(b, a, equal_var=False, nan_policy="omit", alternative="two-sided")

        records.append({
            "metric": metric,
            "reference_group": reference_key,
            "group2": key,
            "group2_index": i,
            "test": test,
            "paired": paired_used,
            "n_used": n,
            "statistic": float(stat) if np.isfinite(stat) else stat,
            "p_value": float(p) if np.isfinite(p) else np.nan,
        })

    pairwise = pd.DataFrame(records)
    if not pairwise.empty:
        valid_mask = pairwise["p_value"].notna()
        adjusted = [np.nan] * len(pairwise)
        adj_valid = holm_adjust(pairwise.loc[valid_mask, "p_value"].tolist())
        for idx, adj in zip(pairwise.index[valid_mask], adj_valid):
            adjusted[idx] = adj
        pairwise["p_adj_holm"] = adjusted
        pairwise["significance"] = pairwise["p_adj_holm"].map(p_to_stars)
        pairwise.loc[pairwise["group2"].astype(str).eq(reference_key), "significance"] = REFERENCE_BAR_TEXT
        pairwise["significant"] = pairwise["p_adj_holm"] < ALPHA

    return pairwise, f"Holm-adjusted tests vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}"


In [ ]:
# ============================================================
# Plotting helpers
# ============================================================

def extract_metric_arrays(selected: pd.DataFrame, metric: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    info = PERTURBATION_EFFECT_METRICS[metric]
    mean_col = info["mean_column"]
    std_col = info["std_column"]
    n_col = info["n_column"]

    if mean_col not in selected.columns:
        raise KeyError(f"Missing metric mean column: {mean_col}")

    means = pd.to_numeric(selected[mean_col], errors="coerce").to_numpy(dtype=float)

    if std_col in selected.columns:
        stds = pd.to_numeric(selected[std_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    else:
        stds = np.zeros_like(means, dtype=float)

    if n_col in selected.columns:
        ns = pd.to_numeric(selected[n_col], errors="coerce").to_numpy(dtype=float)
    else:
        ns = np.full_like(means, np.nan, dtype=float)

    return means, stds, ns


def extract_baseline(full_selection: pd.DataFrame, metric: str) -> tuple[float, float, float]:
    info = PERTURBATION_EFFECT_METRICS[metric]
    mean_col = info["mean_column"]
    std_col = info["std_column"]
    n_col = info["n_column"]

    base = full_selection[full_selection["strategy"].astype(str).eq(NAIVE_BASELINE_ID)].copy()
    if base.empty or mean_col not in base.columns:
        return np.nan, np.nan, np.nan

    means = pd.to_numeric(base[mean_col], errors="coerce").dropna().to_numpy(dtype=float)
    if len(means) == 0:
        return np.nan, np.nan, np.nan

    baseline_mean = float(np.nanmean(means))

    if std_col in base.columns:
        stds = pd.to_numeric(base[std_col], errors="coerce").dropna().to_numpy(dtype=float)
        baseline_std = float(np.nanmean(stds)) if len(stds) else np.nan
    else:
        baseline_std = np.nan

    if n_col in base.columns:
        ns = pd.to_numeric(base[n_col], errors="coerce").dropna().to_numpy(dtype=float)
        baseline_n = float(np.nanmax(ns)) if len(ns) else np.nan
    else:
        baseline_n = np.nan

    return baseline_mean, baseline_std, baseline_n


def compute_axis_limits(
    metric: str,
    means: np.ndarray,
    stds: np.ndarray,
    baseline_mean: float | None = None,
    raw_y_values: list[np.ndarray] | None = None,
) -> tuple[float, float]:
    vals: list[float] = []
    valid = np.isfinite(means)
    vals.extend((means[valid] + np.nan_to_num(stds[valid], nan=0.0)).tolist())
    vals.extend((means[valid] - np.nan_to_num(stds[valid], nan=0.0)).tolist())

    if raw_y_values is not None:
        for arr in raw_y_values:
            arr = np.asarray(arr, dtype=float)
            arr = arr[np.isfinite(arr)]
            vals.extend(arr.tolist())

    if baseline_mean is not None and np.isfinite(baseline_mean):
        vals.append(float(baseline_mean))

    if not vals:
        return 0.0, 1.0

    data_min = float(np.nanmin(vals))
    data_max = float(np.nanmax(vals))
    center = 0.5 * (data_min + data_max)
    span = data_max - data_min

    if metric == "perturbation_effect_pearson":
        min_span = PEARSON_MIN_SPAN
    else:
        min_span = max(abs(center) * MIN_RELATIVE_SPAN, MIN_ABSOLUTE_SPAN)

    if span < min_span:
        data_min = center - 0.5 * min_span
        data_max = center + 0.5 * min_span
        span = min_span

    y_min = data_min - BOTTOM_PADDING_FRACTION * span
    y_max = data_max + TOP_PADDING_FRACTION * span

    # Pearson is bounded, but allow a tiny headroom above 1 for labels.
    if metric == "perturbation_effect_pearson":
        y_min = max(y_min, -1.0)
        y_max = min(y_max, 1.02)

    # For positive error metrics, a slightly negative lower bound avoids visual clipping.
    if metric == "perturbation_effect_rmse" and y_min > 0:
        y_min = max(0.0, y_min)

    return y_min, y_max


def add_naive_baseline_line(
    ax: plt.Axes,
    full_selection: pd.DataFrame,
    metric: str,
    y_min: float,
    y_max: float,
) -> tuple[float, float, float]:
    baseline_mean, baseline_std, baseline_n = extract_baseline(full_selection, metric)

    if not np.isfinite(baseline_mean):
        return baseline_mean, baseline_std, baseline_n

    ax.axhline(
        baseline_mean,
        color=NAIVE_BASELINE_LINE_COLOR,
        linestyle=NAIVE_BASELINE_LINESTYLE,
        linewidth=NAIVE_BASELINE_LINEWIDTH,
        alpha=NAIVE_BASELINE_LINE_ALPHA,
        zorder=1,
    )

    y_range = max(y_max - y_min, abs(y_max) * 0.05, 1e-9)
    text = f"{NAIVE_BASELINE_TEXT}: {final_bar_value(baseline_mean)}"
    if np.isfinite(baseline_std):
        text += f" ± {final_bar_value(baseline_std)}"
    if np.isfinite(baseline_n):
        text += f"; n={int(baseline_n)}"

    ax.text(
        NAIVE_BASELINE_TEXT_X_FRACTION,
        baseline_mean + y_range * NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION,
        text,
        transform=ax.get_yaxis_transform(),
        ha="right",
        va="bottom",
        fontsize=NAIVE_BASELINE_TEXT_SIZE,
        color=NAIVE_BASELINE_TEXT_COLOR,
        clip_on=False,
    )

    return baseline_mean, baseline_std, baseline_n


def plot_metric_barplot(
    selected: pd.DataFrame,
    full_selection: pd.DataFrame,
    seed_df: pd.DataFrame,
    metric: str,
    outdir: str | Path,
    dataset_group_title: str,
    title_suffix: str | None = None,
    ax: plt.Axes | None = None,
    save: bool = True,
) -> dict[str, Any]:
    info = PERTURBATION_EFFECT_METRICS[metric]
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    selected = selected.copy().reset_index(drop=True)
    selected_keys = selected["variant_match_key"].astype(str).tolist()

    y_values, means, stds, ns, used_seed_level = seed_values_for_order(selected, seed_df, metric)
    valid = np.isfinite(means)
    if not valid.any():
        raise RuntimeError(f"All selected variants are NaN for {metric} in {dataset_group_title}.")

    plot_df = selected.copy()
    plot_df["_plot_key"] = selected_keys
    plot_df["_plot_mean"] = means
    plot_df["_plot_std"] = stds
    plot_df["_plot_n"] = ns
    plot_df["_plot_values"] = y_values
    plot_df = plot_df[np.isfinite(plot_df["_plot_mean"].to_numpy(dtype=float))].copy().reset_index(drop=True)

    selected_keys = plot_df["_plot_key"].astype(str).tolist()
    y_values = plot_df["_plot_values"].tolist()
    means = plot_df["_plot_mean"].to_numpy(dtype=float)
    stds = plot_df["_plot_std"].to_numpy(dtype=float)
    ns = plot_df["_plot_n"].to_numpy(dtype=float)
    colors = plot_df["plot_color"].astype(str).tolist()

    baseline_mean, baseline_std, baseline_n, baseline_values = extract_seed_baseline(seed_df, metric)
    if not np.isfinite(baseline_mean):
        baseline_mean, baseline_std, baseline_n = extract_baseline(full_selection, metric)
        baseline_values = np.array([], dtype=float)

    y_min, y_max = compute_axis_limits(
        metric,
        means,
        stds,
        baseline_mean,
        raw_y_values=y_values + ([baseline_values] if len(baseline_values) else []),
    )

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=FIGSIZE)
        created_fig = True
    else:
        fig = ax.figure

    x = np.arange(plot_df.shape[0], dtype=float)

    ax.bar(
        x,
        means,
        yerr=stds,
        width=BAR_WIDTH,
        color=colors,
        alpha=BAR_ALPHA,
        edgecolor="black",
        linewidth=0.7,
        error_kw={
            "ecolor": ERRORBAR_COLOR,
            "elinewidth": ERRORBAR_LINEWIDTH,
            "capsize": ERRORBAR_CAPSIZE,
            "capthick": ERRORBAR_LINEWIDTH,
        },
        zorder=3,
    )

    rng = np.random.default_rng(POINT_RANDOM_SEED)
    seed_to_jitter: dict[Any, float] = {}

    if SHOW_SEED_SCATTERS and used_seed_level:
        for xi, vals, bar_color, key in zip(x, y_values, colors, selected_keys):
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            if len(vals) == 0:
                continue

            # Use seed-specific jitter when seed lines are enabled; otherwise simple random jitter.
            if DRAW_SEED_LINES and not seed_df.empty:
                sub = seed_df.loc[
                    seed_df["variant_match_key"].astype(str).eq(key),
                    ["sampling_seed_for_plot", metric],
                ].dropna()
                xs = []
                plotted_vals = []
                for _, row in sub.iterrows():
                    seed = row["sampling_seed_for_plot"]
                    if seed not in seed_to_jitter:
                        seed_to_jitter[seed] = float(rng.uniform(-POINT_JITTER, POINT_JITTER))
                    xs.append(xi + seed_to_jitter[seed])
                    plotted_vals.append(float(row[metric]))
                vals_to_plot = np.asarray(plotted_vals, dtype=float)
            else:
                xs = np.full(len(vals), xi) + rng.uniform(-POINT_JITTER, POINT_JITTER, size=len(vals))
                vals_to_plot = vals

            ax.scatter(
                xs,
                vals_to_plot,
                s=POINT_SIZE,
                alpha=POINT_ALPHA,
                color=bar_color,
                edgecolor=POINT_EDGE_COLOR,
                linewidth=POINT_EDGE_WIDTH,
                zorder=5,
            )

    if DRAW_SEED_LINES and SHOW_SEED_SCATTERS and used_seed_level and not seed_df.empty:
        pivot = seed_df.pivot_table(index="sampling_seed_for_plot", columns="variant_match_key", values=metric, aggfunc="mean")
        for _, row in pivot[selected_keys].dropna(how="all").iterrows():
            ax.plot(x, row.values, color="#999999", linewidth=0.6, alpha=0.35, zorder=1)

    # Compute significance before final y-limit update, because star labels need extra headroom.
    pairwise = pd.DataFrame()
    mode_used = "no statistical test"
    if SHOW_PAIRWISE_STAR_ANNOTATIONS and used_seed_level and not seed_df.empty:
        pairwise, mode_used = compute_reference_pairwise_statistics(
            seed_df=seed_df,
            metric=metric,
            selected_keys=selected_keys,
            reference_key=REFERENCE_STRATEGY_FOR_TEST,
            stat_mode=STAT_MODE,
        )

    y_range = max(y_max - y_min, abs(y_max) * 0.05, 1e-9)
    bar_value_y_lookup: dict[int, float] = {}

    if SHOW_BAR_VALUE_LABELS:
        offset = max(y_range * BAR_VALUE_OFFSET_FRACTION, y_range * 0.015)
        for i, (xi, mean, sd, vals) in enumerate(zip(x, means, stds, y_values)):
            if not np.isfinite(mean):
                continue
            vals = np.asarray(vals, dtype=float)
            vals = vals[np.isfinite(vals)]
            raw_top = float(np.nanmax(vals)) if len(vals) else mean
            text_y = max(mean + (sd if np.isfinite(sd) else 0.0), raw_top) + offset
            bar_value_y_lookup[i] = text_y
            ax.text(
                xi,
                text_y,
                final_bar_value(mean),
                ha="center",
                va="bottom",
                fontsize=BAR_VALUE_FONT_SIZE,
                color=BAR_VALUE_COLOR,
                rotation=0,
                zorder=8,
                clip_on=False,
            )

    star_y_values: list[float] = []
    if SHOW_PAIRWISE_STAR_ANNOTATIONS and not pairwise.empty:
        star_lookup = {
            str(row["group2"]): str(row.get("significance", "ns"))
            for _, row in pairwise.iterrows()
        }
        text_offset = STAR_TEXT_OFFSET_FRACTION * y_range

        for i, key in enumerate(selected_keys):
            label = star_lookup.get(key, "ns")
            if key == REFERENCE_STRATEGY_FOR_TEST and not SHOW_REFERENCE_BAR_TEXT:
                continue
            if label == "ns" and not STAR_ANNOTATE_NS:
                continue

            vals = np.asarray(y_values[i], dtype=float)
            vals = vals[np.isfinite(vals)]
            raw_top = float(np.nanmax(vals)) if len(vals) else means[i]
            y_bar_top = means[i] + (stds[i] if np.isfinite(stds[i]) else 0.0)
            y_text = max(y_bar_top, raw_top, bar_value_y_lookup.get(i, -np.inf)) + 0.65 * text_offset
            star_y_values.append(y_text)

            ax.text(
                x[i],
                y_text,
                label,
                ha="center",
                va="bottom",
                fontsize=STAR_FONT_SIZE,
                fontweight=STAR_FONT_WEIGHT,
                color=STAR_COLOR,
                clip_on=False,
                zorder=9,
            )

    # Final y limit includes numeric value labels and star annotations.
    tops = [y_max]
    if bar_value_y_lookup:
        tops.extend(bar_value_y_lookup.values())
    if star_y_values:
        tops.extend(star_y_values)
    if tops:
        y_max_final = max(tops) + STAR_Y_EXTRA_FRACTION * y_range
        if metric == "perturbation_effect_pearson":
            y_max_final = min(y_max_final, 1.04)
        y_max = max(y_max, y_max_final)

    ax.set_ylim(y_min, y_max)

    if SHOW_NAIVE_BASELINE_LINE:
        # Draw after ylim is set so the text offset matches the final range.
        if np.isfinite(baseline_mean):
            ax.axhline(
                baseline_mean,
                color=NAIVE_BASELINE_LINE_COLOR,
                linestyle=NAIVE_BASELINE_LINESTYLE,
                linewidth=NAIVE_BASELINE_LINEWIDTH,
                alpha=NAIVE_BASELINE_LINE_ALPHA,
                zorder=1,
            )
            final_range = max(y_max - y_min, abs(y_max) * 0.05, 1e-9)
            text = f"{NAIVE_BASELINE_TEXT}: {final_bar_value(baseline_mean)}"
            if np.isfinite(baseline_std):
                text += f" ± {final_bar_value(baseline_std)}"
            if np.isfinite(baseline_n):
                text += f"; n={int(baseline_n)}"
            ax.text(
                NAIVE_BASELINE_TEXT_X_FRACTION,
                baseline_mean + final_range * NAIVE_BASELINE_TEXT_Y_OFFSET_FRACTION,
                text,
                transform=ax.get_yaxis_transform(),
                ha="right",
                va="bottom",
                fontsize=NAIVE_BASELINE_TEXT_SIZE,
                color=NAIVE_BASELINE_TEXT_COLOR,
                clip_on=False,
            )

    xtick_labels = [
        make_bar_xtick_label(row, n=row.get("_plot_n", np.nan), show_n=True)
        for _, row in plot_df.iterrows()
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, rotation=ROTATE_XTICKS, ha="center")
    ax.set_ylabel(info["ylabel"], fontsize=11)

    if title_suffix is None:
        title_suffix = dataset_group_title
    ax.set_title(
        f"{info['label']} across selected variants\n{title_suffix}",
        fontsize=13 if created_fig else 11,
        weight="bold",
    )

    ax.grid(axis="y", linewidth=0.5, alpha=GRID_ALPHA, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if SHOW_SIGNIFICANCE_NOTE and SHOW_PAIRWISE_STAR_ANNOTATIONS and used_seed_level:
        ax.text(
            0.01,
            0.98,
            f"Stars/ns: Holm-adjusted tests vs {REFERENCE_STRATEGY_FOR_TEST_LABEL}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=8.5,
            color="#555555",
        )

    fig.tight_layout()

    outputs: dict[str, Any] = {
        "metric": metric,
        "dataset_group_title": dataset_group_title,
        "variant_order": plot_df["source_variant_id"].astype(str).tolist() if "source_variant_id" in plot_df.columns else plot_df["variant_id"].astype(str).tolist(),
        "variant_match_keys": selected_keys,
        "means": means,
        "stds": stds,
        "ns": ns,
        "baseline_mean": baseline_mean,
        "baseline_std": baseline_std,
        "baseline_n": baseline_n,
        "used_seed_level": used_seed_level,
        "stat_mode_used": mode_used,
        "pairwise": pairwise,
    }

    if save and created_fig:
        stem = f"{safe_filename(dataset_group_title)}__{info['save_name']}"
        if SAVE_PNG:
            png_path = outdir / f"{stem}.png"
            fig.savefig(png_path, dpi=DPI, bbox_inches="tight")
            outputs["png"] = png_path
            print(f"[Saved] {png_path}")
        if SAVE_SVG:
            svg_path = outdir / f"{stem}.svg"
            fig.savefig(svg_path, bbox_inches="tight")
            outputs["svg"] = svg_path
            print(f"[Saved] {svg_path}")

    if created_fig and not SHOW_FIGURES:
        plt.close(fig)

    return outputs


def make_combined_metric_figure(contexts: list[dict[str, Any]], metric: str, output_root: Path) -> Path | None:
    valid = []
    mean_col = PERTURBATION_EFFECT_METRICS[metric]["mean_column"]

    for ctx in contexts:
        selected = ctx["selected"]
        if mean_col not in selected.columns:
            continue
        vals = pd.to_numeric(selected[mean_col], errors="coerce")
        if vals.notna().sum() > 0:
            valid.append(ctx)

    if not valid:
        print(f"[Skip combined] No data for metric {metric}")
        return None

    n = len(valid)
    ncols = min(COMBINED_NCOLS, n)
    nrows = int(np.ceil(n / ncols))

    width = COMBINED_FIGSIZE_PER_PANEL[0] * ncols
    height = COMBINED_FIGSIZE_PER_PANEL[1] * nrows
    fig, axes = plt.subplots(nrows, ncols, figsize=(width, height), squeeze=False)

    for ax in axes.ravel():
        ax.axis("off")

    for ax, ctx in zip(axes.ravel(), valid):
        ax.axis("on")
        plot_metric_barplot(
            selected=ctx["selected"],
            full_selection=ctx["full_selection"],
            seed_df=ctx.get("seed_df", pd.DataFrame()),
            metric=metric,
            outdir=output_root,
            dataset_group_title=ctx["dataset_group_title"],
            title_suffix=ctx["dataset_group_title"],
            ax=ax,
            save=False,
        )

    fig.suptitle(PERTURBATION_EFFECT_METRICS[metric]["label"], fontsize=16, weight="bold", y=1.005)
    fig.tight_layout()

    output_root.mkdir(parents=True, exist_ok=True)
    out_path = output_root / f"combined__{PERTURBATION_EFFECT_METRICS[metric]['save_name']}.png"
    fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
    print(f"[Saved combined] {out_path}")

    if SAVE_SVG:
        svg_path = output_root / f"combined__{PERTURBATION_EFFECT_METRICS[metric]['save_name']}.svg"
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"[Saved combined] {svg_path}")

    if not SHOW_FIGURES:
        plt.close(fig)

    return out_path

In [ ]:
# ============================================================
# Driver
# ============================================================

def run_selected_perturbation_effect_plots() -> dict[str, Any]:
    paths = candidate_sub_paths()
    if not paths:
        raise RuntimeError(f"No valid dataset/group paths found under root_dir={root_dir}")

    print(f"[Found dataset/group paths] {len(paths)}")
    for p in paths:
        print(f"  - {p}")

    contexts: list[dict[str, Any]] = []
    individual_outputs: list[dict[str, Any]] = []

    for sub_path in paths:
        print("\n" + "=" * 100)
        print(f"[Dataset/group] {get_dataset_group_title(sub_path)}")
        print("=" * 100)

        try:
            ctx = selected_metric_context(sub_path)
        except Exception as exc:
            print(f"[Skip] {sub_path}: {repr(exc)}")
            continue

        contexts.append(ctx)
        outdir = sub_path / "result_analysis" / INDIVIDUAL_OUTPUT_FOLDER_NAME

        for metric, info in PERTURBATION_EFFECT_METRICS.items():
            mean_col = info["mean_column"]
            if mean_col not in ctx["selected"].columns:
                print(f"[Skip metric] {metric}: missing column {mean_col} in {ctx['selection_path']}")
                continue
            if pd.to_numeric(ctx["selected"][mean_col], errors="coerce").notna().sum() == 0:
                print(f"[Skip metric] {metric}: selected values are all NaN")
                continue

            try:
                result = plot_metric_barplot(
                    selected=ctx["selected"],
                    full_selection=ctx["full_selection"],
                    seed_df=ctx.get("seed_df", pd.DataFrame()),
                    metric=metric,
                    outdir=outdir,
                    dataset_group_title=ctx["dataset_group_title"],
                )
                individual_outputs.append(result)
            except Exception as exc:
                print(f"[Skip plot] {ctx['dataset_group_title']} | {metric}: {repr(exc)}")

    combined_outputs = []
    if MAKE_COMBINED_FIGURES and contexts:
        combined_root = root_dir / COMBINED_OUTPUT_FOLDER_NAME
        for metric in PERTURBATION_EFFECT_METRICS:
            path = make_combined_metric_figure(contexts, metric, combined_root)
            if path is not None:
                combined_outputs.append(path)

    print("\n[Done]")
    print(f"Individual plot records: {len(individual_outputs)}")
    print(f"Combined figures: {len(combined_outputs)}")

    return {
        "contexts": contexts,
        "individual_outputs": individual_outputs,
        "combined_outputs": combined_outputs,
    }


if __name__ == "__main__":
    results = run_selected_perturbation_effect_plots()